# Figure 4 — Synthetic Benchmark: Opposing Sampling Densities

Reproduces **Figure 4** from *Density-Reweighted Entropic Optimal Transport*.

**Setup:** Two manifolds M_x and M_y, each composed of two circular arcs (unit circle + radius-2 semi-circle). M_y is M_x shifted by (3, 5). Sampling densities on M_x and M_y are opposing: M_x is 80% on the unit circle, M_y is 80% on the semi-circle. Within-component densities are non-uniform Gaussian mixtures.

**Methods compared:** EOT, DR-EOT (θ=1, Lepski KDE), UOT (λ = γ·ε for γ ∈ {0.1, 1, 10, 100}).

**Hyperparameter selection:** ε selected by golden-section search maximizing MKNN(k=50); KDE bandwidth by global Bootstrap-Lepski.

**Metric:** Symmetric Recall@k = (1/2)(R_k^{X→Y} + R_k^{Y→X}), averaged over 10 replicates.

> **Note:** This experiment is compute-intensive (~hours for 10 replicates). Reduce `N_SIMS` to 1–2 for a quick check.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances
import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.patches import FancyBboxPatch
from scipy.stats import norm
from dataclasses import dataclass
from typing import List, Optional, Tuple
from tqdm import tqdm

from dreot import sinkhorn_eot, sinkhorn_dreot, GlobalBootstrapLepski, WrappedGaussianMixture, Arc, Manifold
from dreot.utils import golden_bandwidth_mnn, knn_intersection, highlight_minmax

# UOT baseline (requires `pip install pot`)
from ot.unbalanced import sinkhorn_knopp_unbalanced as uot_plan

## 1. Manifold Definitions

In [ ]:
# WrappedGaussianMixture, Arc, Manifold imported from dreot above.
# See dreot/models.py for their implementation.

In [ ]:
# ── Paper manifold definitions ───────────────────────────────────────────────
np.random.seed(42)

density1 = WrappedGaussianMixture([0.5*np.pi, 1.5*np.pi], [0.8, 1.0], [0.3, 0.7], a=0, L=2*np.pi)
arc1 = Arc(center=(0, 0), radius=1.0, density=density1, name="M1-Arc1")

density2 = WrappedGaussianMixture([0.25*np.pi, 0.65*np.pi], [0.3, 0.8], [0.6, 0.4], a=0, L=np.pi)
arc2 = Arc(center=(4, 1), radius=2.0, density=density2, name="M1-Arc2")

manifold1 = Manifold(arcs=[arc1, arc2], arc_weights=[0.8, 0.2])

shift = np.array([3, 5])

density3 = WrappedGaussianMixture([0.2*np.pi, 1.3*np.pi], [0.5, 0.9], [0.5, 0.5], a=0, L=2*np.pi)
arc3 = Arc(center=(0+shift[0], 0+shift[1]), radius=1.0, density=density3, name="M2-Arc1")

density4 = WrappedGaussianMixture([0.1*np.pi, 0.7*np.pi], [0.4, 0.6], [0.25, 0.75], a=0, L=np.pi)
arc4 = Arc(center=(4+shift[0], 1+shift[1]), radius=2.0, density=density4, name="M2-Arc2")

manifold2 = Manifold(arcs=[arc3, arc4], arc_weights=[0.2, 0.8])

print(f"Manifold 1 arc weights: {manifold1.arc_weights}")
print(f"Manifold 2 arc weights: {manifold2.arc_weights}  (opposing)")

## 2. Simulation Parameters

In [ ]:
N_SIMS        = 10      # set to 1-2 for a quick sanity check
N = M         = 2000
SEEDS         = 42

K_MNN         = 50
BD_LO, BD_HI  = 0.0, 5.0
MIN_INTERVAL  = 0.10
MAX_ITER_GOLD = 20

DELTA         = 1e-3
MAX_ITER_SINK = 1000

H_RANGE       = (0.5 * np.sqrt(1e-2), 0.5 * np.sqrt(1.0))
N_BANDWIDTHS  = 50
N_BOOTSTRAP   = 20
ALPHA_LEPSKI  = 0.05
C_LEPSKI      = 1
N_JOBS        = 4   # increase if you have more cores
SEED_LEPSKI   = 42

REG_M_FACTORS = [0.1, 1, 10, 100]
K_LIST        = np.arange(5, 101, 5)
METHOD_NAMES  = ["EOT", "DA"] + [f"UOT_{f}" for f in REG_M_FACTORS]

print(f"N_SIMS={N_SIMS}, N=M={N}")
print(f"Methods: {METHOD_NAMES}")

## 3. Run Simulation

In [ ]:
def uot_wrapper(dist, bd, row_sum, col_sum, reg_m_factor, **kwargs):
    return uot_plan(row_sum, col_sum, dist, float(bd),
                    bd * reg_m_factor * np.ones(2), **kwargs)


all_row_results, all_col_results = [], []
np.random.seed(SEEDS)

for sim_idx in range(N_SIMS):
    print(f"\n{'='*60}")
    print(f"Simulation {sim_idx+1}/{N_SIMS}")
    print(f"{'='*60}")

    thetas1, X, arc_ids1 = manifold1.sample(M)
    thetas2, Y, arc_ids2 = manifold2.sample(N)
    dist     = pairwise_distances(X, Y, metric="sqeuclidean", n_jobs=4)
    dist_ref = pairwise_distances(X, Y - shift, metric="sqeuclidean", n_jobs=4)
    W_ref    = -dist_ref   # reference: prefer geometrically close pairs

    row_sum_eot = np.ones((M, 1)) * N
    col_sum_eot = np.ones((N, 1)) * M
    W_dict = {}

    # ── 1. EOT ────────────────────────────────────────────────────────────────
    print("  EOT: bandwidth search ...")
    try:
        best_bd_eot, _ = golden_bandwidth_mnn(
            dist, K_MNN, BD_LO, BD_HI,
            sinkhorn_balance_fn=sinkhorn_eot,
            sinkhorn_kwargs=dict(row_sum=row_sum_eot, col_sum=col_sum_eot,
                                 delta=DELTA, max_iter=MAX_ITER_SINK, check_freq=100,
                                 raise_on_bad_convergence=False),
            max_iter=MAX_ITER_GOLD, min_interval=MIN_INTERVAL,
        )
        row_s, col_s = sinkhorn_eot(
            dist, best_bd_eot, row_sum_eot, col_sum_eot,
            delta=DELTA, max_iter=MAX_ITER_SINK, check_freq=100, raise_on_bad_convergence=False,
        )
        W_dict["EOT"] = row_s * np.exp(-dist / best_bd_eot) * col_s.T
        print(f"    best_bd = {best_bd_eot:.4f}")
    except Exception as e:
        print(f"    EOT failed: {e}")

    # ── 2. DR-EOT (Density-Adjusted) ─────────────────────────────────────────
    print("  DA: Lepski KDE + bandwidth search ...")
    try:
        lepski_X = GlobalBootstrapLepski(X, H_RANGE, N_BANDWIDTHS, N_BOOTSTRAP,
                                         ALPHA_LEPSKI, C_LEPSKI, N_JOBS, SEED_LEPSKI)
        lepski_Y = GlobalBootstrapLepski(Y, H_RANGE, N_BANDWIDTHS, N_BOOTSTRAP,
                                         ALPHA_LEPSKI, C_LEPSKI, N_JOBS, SEED_LEPSKI)
        opt_X = lepski_X.select_bandwidth_global()
        opt_Y = lepski_Y.select_bandwidth_global()
        mu = opt_X.density_estimates.reshape(-1, 1)
        nu = opt_Y.density_estimates.reshape(-1, 1)
        print(f"    h_X={opt_X.optimal_h:.4f}  h_Y={opt_Y.optimal_h:.4f}")

        best_bd_da, _ = golden_bandwidth_mnn(
            dist, K_MNN, BD_LO, BD_HI,
            sinkhorn_balance_fn=sinkhorn_dreot,
            sinkhorn_kwargs=dict(mu=mu, rho=nu, delta=DELTA,
                                 max_iter=MAX_ITER_SINK, check_freq=100,
                                 raise_on_bad_convergence=False),
            max_iter=MAX_ITER_GOLD, min_interval=MIN_INTERVAL,
        )
        row_s_da, col_s_da = sinkhorn_dreot(
            dist, best_bd_da, mu, nu,
            delta=DELTA, max_iter=MAX_ITER_SINK, check_freq=100, raise_on_bad_convergence=False,
        )
        W_dict["DA"] = row_s_da * np.exp(-dist / best_bd_da) * col_s_da.T
        print(f"    best_bd = {best_bd_da:.4f}")
    except Exception as e:
        print(f"    DA failed: {e}")

    # ── 3–6. UOT variants ─────────────────────────────────────────────────────
    for reg_m_factor in REG_M_FACTORS:
        key = f"UOT_{reg_m_factor}"
        print(f"  {key}: bandwidth search ...")
        try:
            best_bd_uot, _ = golden_bandwidth_mnn(
                dist, K_MNN, BD_LO, BD_HI,
                sinkhorn_balance_fn=uot_wrapper,
                sinkhorn_kwargs=dict(row_sum=row_sum_eot.squeeze(),
                                     col_sum=col_sum_eot.squeeze(),
                                     reg_m_factor=reg_m_factor,
                                     reg_type="entropy", numItermax=MAX_ITER_SINK,
                                     stopThr=DELTA),
                returns_plan=True,
                max_iter=MAX_ITER_GOLD, min_interval=MIN_INTERVAL,
            )
            W_dict[key] = uot_wrapper(
                dist, best_bd_uot,
                row_sum_eot.squeeze(), col_sum_eot.squeeze(),
                reg_m_factor, reg_type="entropy",
                numItermax=MAX_ITER_SINK, stopThr=DELTA,
            )
            print(f"    best_bd = {best_bd_uot:.4f}")
        except Exception as e:
            print(f"    {key} failed: {e}")

    # ── Evaluate ──────────────────────────────────────────────────────────────
    all_row_results.append(knn_intersection(W_dict, W_ref, K_LIST, axis=1, mode="detailed",
                                             labels_X=arc_ids1, labels_Y=arc_ids2))
    all_col_results.append(knn_intersection(W_dict, W_ref, K_LIST, axis=0, mode="detailed",
                                             labels_X=arc_ids1, labels_Y=arc_ids2))
    print("  Done.")

print("\nAll simulations complete.")

## 4. Aggregate Results

In [ ]:
k_cols = [f"k={k}" for k in K_LIST]
k_arr  = K_LIST.astype(float)

per_sim_recall = []
for row_res, col_res in zip(all_row_results, all_col_results):
    if "overall" not in row_res or "overall" not in col_res:
        continue
    df_row = row_res["overall"].reindex(index=METHOD_NAMES, columns=k_cols).values.astype(float)
    df_col = col_res["overall"].reindex(index=METHOD_NAMES, columns=k_cols).values.astype(float)
    per_sim_recall.append(0.5 * (df_row / k_arr + df_col / k_arr))

per_sim_recall = np.stack(per_sim_recall, axis=0)
recall_mean = np.nanmean(per_sim_recall, axis=0)
recall_std  = np.nanstd(per_sim_recall, axis=0, ddof=1)

recall_df = pd.DataFrame(recall_mean, index=METHOD_NAMES, columns=k_cols).round(4)
print("Mean Recall@k (symmetric):")
display(highlight_minmax(recall_df))

## 5. Figure 4 — Manifold plot + Recall@k

In [ ]:
METHOD_COLORS = {
    "EOT":     "#4878CF",
    "DA":      "#D65F5F",
    "UOT_0.1": "#6ACC65",
    "UOT_1":   "#B47CC7",
    "UOT_10":  "#C4AD66",
    "UOT_100": "#77BEDB",
}
METHOD_LABEL = {
    "EOT":     "EOT",
    "DA":      "DR-EOT (ours)",
    "UOT_0.1": r"UOT ($\gamma=0.1$)",
    "UOT_1":   r"UOT ($\gamma=1$)",
    "UOT_10":  r"UOT ($\gamma=10$)",
    "UOT_100": r"UOT ($\gamma=100$)",
}

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.labelsize": 13, "axes.titlesize": 12,
    "xtick.labelsize": 12, "ytick.labelsize": 12,
    "legend.fontsize": 12, "axes.linewidth": 0.8,
    "lines.linewidth": 2.0, "figure.dpi": 150,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.subplots_adjust(wspace=0.35)

# ── Left: manifold visualization ───────────────────────────────────────────
ax0 = axes[0]
vmin = min(manifold1.get_density_range()[0], manifold2.get_density_range()[0])
vmax = max(manifold1.get_density_range()[1], manifold2.get_density_range()[1])
manifold1.plot_arcs(ax=ax0, num_points=1000, s=10, add_colorbar=False, vmin=vmin, vmax=vmax)
sc2 = manifold2.plot_arcs(ax=ax0, num_points=1000, s=10, add_colorbar=False, vmin=vmin, vmax=vmax)
ax0.set_xticks([]); ax0.set_yticks([])

divider = make_axes_locatable(ax0)
cax = divider.append_axes("right", size="5%", pad=0.05)
cb = fig.colorbar(sc2, cax=cax, label="Sampling Density")
cb.set_ticks([])

for box, label in [
    (dict(x0=0.02, y0=0.02, w=0.68, h=0.45), r"$\mathcal{M}_x$"),
    (dict(x0=0.30, y0=0.52, w=0.68, h=0.45), r"$\mathcal{M}_y$"),
]:
    rect = FancyBboxPatch((box["x0"], box["y0"]), box["w"], box["h"],
                          boxstyle="round,pad=0.01", linewidth=0.8, linestyle="--",
                          edgecolor="#666666", facecolor="none",
                          transform=ax0.transAxes, clip_on=False, zorder=5)
    ax0.add_patch(rect)
    ax0.text(box["x0"] + 0.005, box["y0"] + 0.4, label,
             transform=ax0.transAxes, fontsize=15, va="bottom", ha="left",
             color="#444444", zorder=6)

# ── Right: Recall@k ────────────────────────────────────────────────────────
ax1 = axes[1]
for m_name in METHOD_NAMES:
    idx  = METHOD_NAMES.index(m_name)
    mean = recall_mean[idx]
    if np.all(np.isnan(mean)):
        continue
    ax1.plot(K_LIST, mean, "o-",
             color=METHOD_COLORS.get(m_name, "#333"),
             linewidth=3 if m_name == "DA" else 2,
             markersize=6, markerfacecolor="white", markeredgewidth=1.4,
             zorder=3 if m_name == "DA" else 2,
             label=METHOD_LABEL.get(m_name, m_name))

ax1.set_xlabel(r"$k$")
ax1.set_ylabel(r"$\overline{R_k}$")
ax1.set_xlim(K_LIST[0] - 0.5, K_LIST[-1] + 0.5)
ax1.grid(True, which="major", linestyle=":", linewidth=0.5, color="#cccccc", zorder=0)
ax1.legend(loc="upper left", frameon=False, handlelength=2.5, bbox_to_anchor=(-0.01, 1.02))

fig.text(0.01, 1.0, "(a)", fontsize=12, fontweight="bold", va="top")
fig.text(0.47, 1.0, "(b)", fontsize=12, fontweight="bold", va="top")

plt.tight_layout(pad=0.4)
plt.savefig("fig4_simulation.pdf")
plt.show()
print("Saved fig4_simulation.pdf")